# Qwen3-Embedding-4B for new postings, on one Colab TPU v5e chip

Embeds the chunks in `pending.json.gz`, written by `make export-chunk-vectors-pending`, exactly as the whole index was embedded on 2026-11-02 (bfloat16, left padding, last-token pooling, unit length), so the new vectors compare with the stored ones.

1. **Kernel: Colab, TPU v5e-1.** In VS Code: Select Kernel → Colab. In the browser: Runtime → Change runtime type.
2. **Put `pending.json.gz` in `/content/` or anywhere in My Drive.** Cell 2 finds it, and asks to mount Drive if it needs to.
3. **Run All.** About 3 minutes to set up, then about a minute per 8,000 chunks.
4. **Cell 6 saves `qwen4b_batch_f16.npy`** to My Drive → `jobrunner_qwen_batches/<when the export was made>/` (and `/content/`). Put it in the export's folder on the Mac and run the import command the export printed.

Only public posting text is in this notebook.

## 1. Check the runtime has a TPU and internet

In [ ]:
import importlib.metadata as md
import json
import os
import platform
import shutil
import sys
import urllib.request

facts = {
    "python": sys.version.split()[0],
    "machine": platform.machine(),
    "cpus": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9),
}
for key in ("TPU_ACCELERATOR_TYPE", "ACCELERATOR_TYPE", "TPU_WORKER_ID"):
    if os.environ.get(key):
        facts[key] = os.environ[key]
facts["dev_accel"] = sorted(p for p in os.listdir("/dev") if p.startswith("accel"))
facts["dev_vfio"] = (
    sorted(p for p in os.listdir("/dev/vfio") if p != "vfio") if os.path.isdir("/dev/vfio") else []
)
facts["nvidia"] = bool(shutil.which("nvidia-smi"))
try:
    req = urllib.request.Request(
        "http://metadata.google.internal/computeMetadata/v1/instance/attributes/accelerator-type",
        headers={"Metadata-Flavor": "Google"},
    )
    facts["metadata_accelerator"] = urllib.request.urlopen(req, timeout=5).read().decode()[:40]
except Exception as exc:
    facts["metadata_accelerator"] = type(exc).__name__
for name in ("torch", "torch_xla", "jax", "libtpu", "transformers"):
    try:
        facts[name] = md.version(name)
    except Exception:
        facts[name] = None
for label, url in (
    ("hub", "https://huggingface.co/api/models/Qwen/Qwen3-Embedding-4B"),
    ("pypi", "https://pypi.org/simple/torch-tpu/"),
):
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            facts[label] = r.status
    except Exception as exc:
        facts[label] = type(exc).__name__
facts["disk_free_gb"] = round(shutil.disk_usage("/tmp").free / 1e9, 1)
print("FACTS " + json.dumps(facts), flush=True)

if not (facts.get("dev_accel") or facts.get("dev_vfio")):
    raise RuntimeError(
        "No TPU on this runtime: Runtime → Change runtime type → TPU v5e-1, then Run all again."
    )
if facts.get("pypi") != 200 or facts.get("hub") != 200:
    raise RuntimeError("PyPI or Hugging Face is not reachable from this runtime.")
print("TPU found:", facts.get("dev_accel") or facts.get("dev_vfio"))

## 2. Read pending.json.gz

In [ ]:
import glob
import gzip
import json
import os
import time

NAME = "pending.json.gz"
DRIVE = "/content/drive/MyDrive"


def candidates():
    """Every copy in /content or My Drive up to three folders deep, newest first."""
    patterns = ["/content/" + NAME] + [DRIVE + "/" + "*/" * depth + NAME for depth in range(4)]
    found = set()
    for pattern in patterns:
        found |= {path for path in glob.glob(pattern) if os.path.isfile(path)}
    return sorted(found, key=os.path.getmtime, reverse=True)


hits = candidates()
if not hits and not os.path.isdir(DRIVE):
    from google.colab import drive

    drive.mount("/content/drive")
    for _ in range(6):  # a file uploaded a moment ago can take a few seconds to appear
        hits = candidates()
        if hits:
            break
        time.sleep(5)
if not hits:
    raise FileNotFoundError(
        f"{NAME} is not in /content or My Drive. Put it in one of them, then run this cell again."
    )
src = hits[0]
if len(hits) > 1:
    print("several copies; using the newest:", src)
with gzip.open(src, "rt", encoding="utf-8") as handle:
    data = json.load(handle)
chunks = [text for posting in data["postings"] for text in posting["chunks"]]
with open("/tmp/batch_chunks.json", "w") as handle:
    json.dump({"chunks": chunks}, handle)
MADE = str(data.get("made", "unknown"))[:16].replace(":", "")
print(
    f"reading {src}: {len(data['postings']):,} postings, {len(chunks):,} chunks, "
    f"made {data.get('made')}; "
    f"about {len(chunks) / 137 / 60:.1f} min to embed"
)

## 3. Install torch_tpu in its own environment and download the model

In [ ]:
import os
import subprocess
import sys
import time

PY = "/tmp/tt/bin/python"


def sh(args, timeout=1800):
    done = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if done.returncode != 0:
        raise RuntimeError((done.stderr or done.stdout).strip()[-1500:])
    return done.stdout


t = time.time()
if not os.path.exists(PY):
    try:
        sh([sys.executable, "-m", "venv", "/tmp/tt"])
    except Exception:
        sh([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        sh([sys.executable, "-m", "virtualenv", "/tmp/tt"])
sh([PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
sh(
    [
        PY,
        "-m",
        "pip",
        "install",
        "-q",
        "torch_tpu==0.2.0",
        "transformers>=4.51",
        "numpy",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cpu",
    ]
)
print(
    "installed in",
    round(time.time() - t),
    "s:",
    sh(
        [
            PY,
            "-c",
            "import importlib.metadata as m; "
            "print({n: m.version(n) for n in ('torch', 'torch_tpu', 'transformers')})",
        ]
    ).strip(),
)
print(
    "torch_tpu sees:",
    sh(
        [
            PY,
            "-c",
            "import torch; print(torch.accelerator.device_count(), 'chip(s),', "
            "torch.accelerator.current_accelerator())",
        ]
    ).strip(),
)
t = time.time()
sh([PY, "-c", "from huggingface_hub import snapshot_download as s; s('Qwen/Qwen3-Embedding-4B')"])
print("downloaded the model in", round(time.time() - t), "s")

## 4. The embedding script (the whole-index run's settings)

In [ ]:
%%writefile /tmp/tt_batch.py
import json, time
import numpy as np
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

# The same model and settings as the whole-index run of 2026-11-02. Changing
# any of them makes these vectors incomparable with the stored ones.
MODEL = "Qwen/Qwen3-Embedding-4B"
B = 64
PROGRESS_EVERY = 0.1

chunks = json.load(open("/tmp/batch_chunks.json"))["chunks"]
dev = torch.device("tpu")
dtype_kw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")


def say(msg):
    print(msg, flush=True)


def bucket(n):
    return 128 if n <= 128 else 256 if n <= 256 else 512


def build(attn, compiled):
    t = time.perf_counter()
    model = AutoModel.from_pretrained(MODEL, attn_implementation=attn, **{dtype_kw: torch.bfloat16})
    model = model.to(dev).eval()
    load_s = time.perf_counter() - t
    return model, (torch.compile(model, dynamic=False) if compiled else model), load_s


def embed(run, texts, lens, progress=False):
    order = sorted(range(len(texts)), key=lambda i: lens[i])
    out = np.zeros((len(texts), HIDDEN), dtype=np.float16)
    i, done, next_report, t0 = 0, 0, PROGRESS_EVERY, time.perf_counter()
    while i < len(order):
        size = bucket(lens[order[i]])
        idx = []
        while i < len(order) and len(idx) < B and bucket(lens[order[i]]) == size:
            idx.append(order[i]); i += 1
        batch = [texts[j] for j in idx] + [""] * (B - len(idx))
        enc = tok(batch, padding="max_length", truncation=True, max_length=size, return_tensors="pt")
        with torch.no_grad():
            hidden = run(input_ids=enc["input_ids"].to(dev),
                         attention_mask=enc["attention_mask"].to(dev)).last_hidden_state
            vec = torch.nn.functional.normalize(hidden[:, -1, :].float(), p=2, dim=1)
        out[idx] = vec[: len(idx)].cpu().numpy().astype(np.float16)
        done += len(idx)
        if progress and done / len(texts) >= next_report:
            spent = time.perf_counter() - t0
            say(f"  {done:,} of {len(texts):,} chunks ({done / len(texts):.0%}) in {spent / 60:.1f} min")
            next_report += PROGRESS_EVERY
    return out


lengths = [len(x) for x in tok(chunks, truncation=True, max_length=512)["input_ids"]]
warm = {}
for i, n in enumerate(lengths):
    warm.setdefault(bucket(n), i)
result = {"chunks": len(chunks), "tokens": sum(lengths)}
for attn, compiled in (("sdpa", True), ("eager", True), ("sdpa", False)):
    try:
        model, run, load_s = build(attn, compiled)
        HIDDEN = model.config.hidden_size
        t = time.perf_counter()
        embed(run, [chunks[i] for i in warm.values()], [lengths[i] for i in warm.values()])
        result.update(attn=attn, compiled=compiled, load_s=round(load_s), compile_s=round(time.perf_counter() - t))
        say(f"ready: attention {attn}, torch.compile {compiled}")
        break
    except Exception as exc:
        say(f"attention {attn}, torch.compile {compiled} failed: {type(exc).__name__}: {str(exc)[:200]}")
else:
    print("RESULT " + json.dumps({**result, "error": "no configuration ran"}), flush=True)
    raise SystemExit(1)

torch.accelerator.synchronize()
t = time.perf_counter()
docs = embed(run, chunks, lengths, progress=True)
torch.accelerator.synchronize()
result.update(encode_s=round(time.perf_counter() - t), finite=bool(np.isfinite(docs).all()),
              shape=list(docs.shape))
np.save("/content/qwen4b_batch_f16.npy", docs)
print("RESULT " + json.dumps(result), flush=True)

## 5. Embed every chunk

In [ ]:
import json
import os

env = dict(os.environ, TPU_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
proc = subprocess.Popen(
    [PY, "/tmp/tt_batch.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
)
result = None
for line in proc.stdout:
    if line.startswith("RESULT "):
        result = json.loads(line[7:])
    else:
        print(line, end="")
proc.wait()
ok = bool(result and result.get("finite") and result.get("shape", [0])[0] == len(chunks))
print(
    f"embedded {result['chunks']:,} chunks in {result['encode_s'] / 60:.1f} min"
    if ok
    else f"failed: {result}"
)

## 6. Save the vectors

In [ ]:
import shutil

if not ok:
    raise RuntimeError("The embedding did not finish cleanly (cell 5); nothing was saved.")
out = None
try:
    if not os.path.isdir(DRIVE):
        from google.colab import drive

        drive.mount("/content/drive")
    out = f"{DRIVE}/jobrunner_qwen_batches/{MADE}"
    os.makedirs(out, exist_ok=True)
    shutil.copy("/content/qwen4b_batch_f16.npy", out)
except Exception as exc:
    print("Could not save to Drive:", type(exc).__name__, str(exc)[:200])
size = os.path.getsize("/content/qwen4b_batch_f16.npy") / 1e6
print(f"qwen4b_batch_f16.npy ({size:.1f} MB) is in /content/" + (f" and {out}/" if out else ""))
print("Put it in the export's folder on the Mac, then run the import command the export printed.")